In [10]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split 
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

def create_sequences_fix(data, timesteps, predict_day):
    X, y = [], []
    for i in range(len(data) - timesteps - predict_day):
        X.append(data.iloc[i:(i + timesteps)])
        y.append(data.iloc[(i + timesteps):(i + timesteps + predict_day)])
        #y.append(data.iloc[(i + timesteps + predict_day)])
    return np.array(X), np.array(y)

def create_sequences_fix_test(data, timesteps, predict_day):
    X, y = [], []
    for i in range(0, len(data) - timesteps - predict_day, predict_day):
        X.append(data.iloc[i:(i + timesteps)])
        y.append(data.iloc[(i + timesteps):(i + timesteps + predict_day)])
    return np.array(X), np.array(y)

from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd

import datetime
import calendar

def days(year):
    return pd.Timestamp(year, 12, 31).dayofyear

def yearly_comparison(prediction, truelabel, pred_len):
    yearly_results = {year: {"rmse": [], "mae": [], "mape": []} for year in range(2009, 2022)}
    
    for day in range(pred_len):
        day_predictions = prediction[:, day]
        day_true_values = truelabel[:, day]
        year = 2009
        start_index = 0
        end_index = 0
        
        while year <= 2021:
            day_count = days(year)
            end_index += day_count
            year_prediction = day_predictions[start_index:end_index]
            year_true_value = day_true_values[start_index:end_index]
            start_index += day_count

            # Calculate metrics for the current year
            rmse1 = mean_squared_error(year_true_value, year_prediction, squared=False)
            mae1 = mean_absolute_error(year_true_value, year_prediction)
            mape1 = mean_absolute_percentage_error(year_true_value, year_prediction) * 100

            yearly_results[year]["rmse"].append(rmse1)
            yearly_results[year]["mae"].append(mae1)
            yearly_results[year]["mape"].append(mape1)
            year += 1

    # Calculate yearly averages
    yearly_averages = {year: {
        "avg_rmse": np.mean(metrics["rmse"]),
        "avg_mae": np.mean(metrics["mae"]),
        "avg_mape": np.mean(metrics["mape"])
    } for year, metrics in yearly_results.items()}
    
    print("Yearly Comparison Results:")
    for year, metrics in yearly_averages.items():
        print(f"Year {year} - RMSE: {metrics['avg_rmse']:.4f}, MAE: {metrics['avg_mae']:.4f}, MAPE: {metrics['avg_mape']:.2f}%")
    
    return yearly_averages

def yearly_comparison_onday(prediction, truelabel, pred_len):
    yearly_results = {year: {"rmse": [], "mae": [], "mape": []} for year in range(2009, 2022)}
    
    day_predictions = prediction[:, pred_len]
    day_true_values = truelabel[:, pred_len]
    year = 2009
    start_index = 0
    end_index = 0
    
    while year <= 2021:
        day_count = days(year)
        end_index += day_count
        year_prediction = day_predictions[start_index:end_index]
        year_true_value = day_true_values[start_index:end_index]
        start_index += day_count

        # Calculate metrics for the current year
        rmse1 = mean_squared_error(year_true_value.flatten(), year_prediction.flatten(), squared=False)
        mae1 = mean_absolute_error(year_true_value.flatten(), year_prediction.flatten())
        mape1 = mean_absolute_percentage_error(year_true_value.flatten(), year_prediction.flatten()) * 100

        yearly_results[year]["rmse"].append(rmse1)
        yearly_results[year]["mae"].append(mae1)
        yearly_results[year]["mape"].append(mape1)
        year += 1

    # Calculate yearly averages
    yearly_averages = {year: {
        "avg_rmse": np.mean(metrics["rmse"]),
        "avg_mae": np.mean(metrics["mae"]),
        "avg_mape": np.mean(metrics["mape"])
    } for year, metrics in yearly_results.items()}
    
    print("Yearly Comparison Results:")
    for year, metrics in yearly_averages.items():
        print(f"Year {year} - RMSE: {metrics['avg_rmse']:.4f}, MAE: {metrics['avg_mae']:.4f}, MAPE: {metrics['avg_mape']:.2f}%")
    
    return yearly_averages

import torch
import torch.nn as nn  
import torch.nn.functional as F
import torch.fft
from layers.Embed import DataEmbedding
from layers.Conv_Blocks import Inception_Block_V1


def FFT_for_Period(x, k=2):
    # [B, T, C]
    xf = torch.fft.rfft(x, dim=1)
    # find period by amplitudes
    frequency_list = abs(xf).mean(0).mean(-1)
    frequency_list[0] = 0
    _, top_list = torch.topk(frequency_list, k)
    top_list = top_list.detach().cpu().numpy()
    period = x.shape[1] // top_list
    return period, abs(xf).mean(-1)[:, top_list]

class TimesBlock(nn.Module):
    def __init__(self, configs):
        super(TimesBlock, self).__init__()
        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len
        self.k = configs.top_k
        # parameter-efficient design
        self.conv = nn.Sequential(
            Inception_Block_V1(configs.d_model, configs.d_ff,
                               num_kernels=configs.num_kernels),
            nn.GELU(),
            Inception_Block_V1(configs.d_ff, configs.d_model,
                               num_kernels=configs.num_kernels)
        )

    def forward(self, x):
        B, T, N = x.size()
        period_list, period_weight = FFT_for_Period(x, self.k)

        res = []
        for i in range(self.k):
            period = period_list[i]
            # padding
            if (self.seq_len + self.pred_len) % period != 0:
                length = (
                                 ((self.seq_len + self.pred_len) // period) + 1) * period
                padding = torch.zeros([x.shape[0], (length - (self.seq_len + self.pred_len)), x.shape[2]]).to(x.device)
                out = torch.cat([x, padding], dim=1)
            else:
                length = (self.seq_len + self.pred_len)
                out = x
            # reshape
            out = out.reshape(B, length // period, period,
                              N).permute(0, 3, 1, 2).contiguous()
            # 2D conv: from 1d Variation to 2d Variation
            out = self.conv(out)
            # reshape back
            out = out.permute(0, 2, 3, 1).reshape(B, -1, N)
            res.append(out[:, :(self.seq_len + self.pred_len), :])
        res = torch.stack(res, dim=-1)
        # adaptive aggregation
        period_weight = F.softmax(period_weight, dim=1)
        period_weight = period_weight.unsqueeze(
            1).unsqueeze(1).repeat(1, T, N, 1)
        res = torch.sum(res * period_weight, -1)
        # residual connection
        res = res + x
        return res


class Model(nn.Module):
    """
    Paper link: https://openreview.net/pdf?id=ju_Uqw384Oq
    """

    def __init__(self, configs):
        super(Model, self).__init__()
        self.configs = configs
        self.task_name = configs.task_name
        self.seq_len = configs.seq_len
        self.label_len = configs.label_len
        self.pred_len = configs.pred_len
        self.model = nn.ModuleList([TimesBlock(configs)
                                    for _ in range(configs.e_layers)])
        self.enc_embedding = DataEmbedding(configs.enc_in, configs.d_model, configs.embed, configs.freq,
                                           configs.dropout)
        self.layer = configs.e_layers
        self.layer_norm = nn.LayerNorm(configs.d_model)
        if self.task_name == 'long_term_forecast' or self.task_name == 'short_term_forecast':
            self.predict_linear = nn.Linear(
                self.seq_len, self.pred_len + self.seq_len)
            self.projection = nn.Linear(
                configs.d_model, configs.c_out, bias=True)
        if self.task_name == 'imputation' or self.task_name == 'anomaly_detection':
            self.projection = nn.Linear(
                configs.d_model, configs.c_out, bias=True)
        if self.task_name == 'classification':
            self.act = F.gelu
            self.dropout = nn.Dropout(configs.dropout)
            self.projection = nn.Linear(
                configs.d_model * configs.seq_len, configs.num_class)

    def forecast(self, x_enc, x_mark_enc, x_dec, x_mark_dec):
        # Normalization from Non-stationary Transformer
        means = x_enc.mean(1, keepdim=True).detach()
        x_enc = x_enc - means
        stdev = torch.sqrt(
            torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)
        x_enc /= stdev

        # embedding
        enc_out = self.enc_embedding(x_enc, x_mark_enc)  # [B,T,C]
        enc_out = self.predict_linear(enc_out.permute(0, 2, 1)).permute(
            0, 2, 1)  # align temporal dimension
        # TimesNet
        for i in range(self.layer):
            enc_out = self.layer_norm(self.model[i](enc_out))
        # porject back
        dec_out = self.projection(enc_out)

        # De-Normalization from Non-stationary Transformer
        dec_out = dec_out * \
                  (stdev[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1))
        dec_out = dec_out + \
                  (means[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1))
        return dec_out

    def imputation(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask):
        # Normalization from Non-stationary Transformer
        means = torch.sum(x_enc, dim=1) / torch.sum(mask == 1, dim=1)
        means = means.unsqueeze(1).detach()
        x_enc = x_enc - means
        x_enc = x_enc.masked_fill(mask == 0, 0)
        stdev = torch.sqrt(torch.sum(x_enc * x_enc, dim=1) /
                           torch.sum(mask == 1, dim=1) + 1e-5)
        stdev = stdev.unsqueeze(1).detach()
        x_enc /= stdev

        # embedding
        enc_out = self.enc_embedding(x_enc, x_mark_enc)  # [B,T,C]
        # TimesNet
        for i in range(self.layer):
            enc_out = self.layer_norm(self.model[i](enc_out))
        # porject back
        dec_out = self.projection(enc_out)

        # De-Normalization from Non-stationary Transformer
        dec_out = dec_out * \
                  (stdev[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1))
        dec_out = dec_out + \
                  (means[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1))
        return dec_out

    def anomaly_detection(self, x_enc):
        # Normalization from Non-stationary Transformer
        means = x_enc.mean(1, keepdim=True).detach()
        x_enc = x_enc - means
        stdev = torch.sqrt(
            torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)
        x_enc /= stdev

        # embedding
        enc_out = self.enc_embedding(x_enc, None)  # [B,T,C]
        # TimesNet
        for i in range(self.layer):
            enc_out = self.layer_norm(self.model[i](enc_out))
        # porject back
        dec_out = self.projection(enc_out)

        # De-Normalization from Non-stationary Transformer
        dec_out = dec_out * \
                  (stdev[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1))
        dec_out = dec_out + \
                  (means[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1))
        return dec_out

    def classification(self, x_enc, x_mark_enc):
        # embedding
        enc_out = self.enc_embedding(x_enc, None)  # [B,T,C]
        # TimesNet
        for i in range(self.layer):
            enc_out = self.layer_norm(self.model[i](enc_out))

        # Output
        # the output transformer encoder/decoder embeddings don't include non-linearity
        output = self.act(enc_out)
        output = self.dropout(output)
        # zero-out padding embeddings
        output = output * x_mark_enc.unsqueeze(-1)
        # (batch_size, seq_length * d_model)
        output = output.reshape(output.shape[0], -1)
        output = self.projection(output)  # (batch_size, num_classes)
        return output

    def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask=None):
        if self.task_name == 'long_term_forecast' or self.task_name == 'short_term_forecast':
            dec_out = self.forecast(x_enc, x_mark_enc, x_dec, x_mark_dec)
            return dec_out[:, -self.pred_len:, :]  # [B, L, D]
        if self.task_name == 'imputation':
            dec_out = self.imputation(
                x_enc, x_mark_enc, x_dec, x_mark_dec, mask)
            return dec_out  # [B, L, D]
        if self.task_name == 'anomaly_detection':
            dec_out = self.anomaly_detection(x_enc)
            return dec_out  # [B, L, D]
        if self.task_name == 'classification':
            dec_out = self.classification(x_enc, x_mark_enc)
            return dec_out  # [B, N]
        return None



In [11]:
import os
import pickle
import numpy as np
import pandas as pd


def load_and_process_raw_test(preprocess_method_path, raw_test_csv_path):
    """
    Load the saved preprocessing method and raw test CSV,
    then generate TestX/Testy for model inference.

    Required files for each prediction length:
        F10.7_preprocess_method_seq30_pred{day}.pkl
        F10.7_raw_test_set_seq30_pred{day}.csv

    Output:
        TestX: [N, seq_len, 1]
        Testy: [N, pred_len, 1]
        scaler: fitted MinMaxScaler, used for inverse_transform
    """
    with open(preprocess_method_path, "rb") as f:
        preprocess_obj = pickle.load(f)

    value_col = preprocess_obj["value_col"]
    timesteps = preprocess_obj["timesteps"]
    predict_day = preprocess_obj["predict_day"]
    scaler = preprocess_obj["scaler"]

    raw_test_df = pd.read_csv(raw_test_csv_path).fillna(0)

    if "date" in raw_test_df.columns:
        raw_test_df["date"] = pd.to_datetime(raw_test_df["date"])
        raw_test_df = raw_test_df.sort_values("date").reset_index(drop=True)

    raw_test_values = raw_test_df[value_col]

    test_scaled = scaler.transform(
        raw_test_values.values.reshape(-1, 1)
    ).flatten()

    TestX, Testy = create_sequences_fix(
        pd.Series(test_scaled),
        timesteps,
        predict_day
    )

    TestX = TestX.reshape(TestX.shape[0], TestX.shape[1], 1)
    Testy = Testy.reshape(Testy.shape[0], Testy.shape[1], 1)

    return TestX, Testy, scaler, raw_test_df


In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
)


# =========================================================
# Settings
# =========================================================

EVAL_DAYS = [1, 27, 45, 60]
DATA_NAME = "F30"
SEQ_LEN = 30
BATCH_SIZE = 32

# Folder containing saved preprocessing methods and raw test csv files
# You should have files like:
#   F10.7_preprocess_method_seq30_pred1.pkl
#   F10.7_raw_test_set_seq30_pred1.csv
#   F10.7_preprocess_method_seq30_pred27.pkl
#   F10.7_raw_test_set_seq30_pred27.csv
#   F10.7_preprocess_method_seq30_pred45.pkl
#   F10.7_raw_test_set_seq30_pred45.csv
#   F10.7_preprocess_method_seq30_pred60.pkl
#   F10.7_raw_test_set_seq30_pred60.csv
SHARED_PREPROCESS_DIR = "./data"

# Folder containing trained models
MODEL_DIR = "."


def build_dataloader(X, y, batch_size=32, shuffle=False):
    dataset = TensorDataset(
        torch.Tensor(X),
        torch.Tensor(y)
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def load_sinet_model(model_name, configs, device):
    """
    Robustly load SINet / TimesNet model.

    Supports both:
    1. full model saved by torch.save(model, path)
    2. state_dict saved by torch.save(model.state_dict(), path)
    """
    ckpt = torch.load(model_name, map_location=device)

    if isinstance(ckpt, dict):
        model = Model(configs).to(device)
        model.load_state_dict(ckpt)
    else:
        model = ckpt.to(device)

    model.eval()
    return model


def get_shared_test_paths(data_name, seq_len, pred_len):
    preprocess_method_path = os.path.join(
        SHARED_PREPROCESS_DIR,
        f"{data_name}_preprocess.pkl"
    )

    raw_test_csv_path = os.path.join(
        SHARED_PREPROCESS_DIR,
        f"{data_name}_test.csv"
    )

    if not os.path.exists(preprocess_method_path):
        raise FileNotFoundError(
            f"Missing preprocessing method file:\n{preprocess_method_path}\n"
            f"You need to save one preprocessing .pkl for pred_len={pred_len}."
        )

    if not os.path.exists(raw_test_csv_path):
        raise FileNotFoundError(
            f"Missing raw test csv file:\n{raw_test_csv_path}\n"
            f"You need to save one raw test csv for pred_len={pred_len}."
        )

    return preprocess_method_path, raw_test_csv_path


# =========================================================
# Reproduce evaluation for 1, 27, 45, 60 days
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

day_results = {
    day: {"rmse": [], "mae": [], "mape": []}
    for day in EVAL_DAYS
}

for day in EVAL_DAYS:
    model_name = os.path.join(MODEL_DIR, f"SINet_{day}_interval_F10.pth")

    seq_len = SEQ_LEN
    pred_len = day

    print("=" * 80)
    print(f"Data: {DATA_NAME}, Sequence Length: {seq_len}, Predict Length: {pred_len}")
    print(f"Loading model: {model_name}")

    preprocess_method_path, raw_test_csv_path = get_shared_test_paths(
        DATA_NAME,
        seq_len,
        pred_len
    )

    print(f"Loading preprocessing: {preprocess_method_path}")
    print(f"Loading raw test csv : {raw_test_csv_path}")

    TestX, Testy, scaler, raw_test_df = load_and_process_raw_test(
        preprocess_method_path=preprocess_method_path,
        raw_test_csv_path=raw_test_csv_path
    )

    print("TestX shape:", TestX.shape)
    print("Testy shape:", Testy.shape)
    print("Raw test date range:", raw_test_df["date"].min(), "to", raw_test_df["date"].max())

    test_loader = build_dataloader(
        TestX,
        Testy,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    sl = seq_len
    pl = pred_len

    class Config:
        seq_len = sl
        pred_len = pl
        enc_in = 1
        c_out = 1
        d_model = 32
        d_ff = 64
        num_kernels = 6
        e_layers = 2
        top_k = 3
        task_name = "long_term_forecast"
        dropout = 0.3
        label_len = 1
        freq = "d"
        embed = "timeF"

    configs = Config()
    model = load_sinet_model(model_name, configs, device)

    test_predictions = []
    test_targets = []

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test = x_test.to(device)
            y_test = y_test.to(device)

            outputs = model(x_test, None, None, None)

            test_predictions.append(outputs.cpu().numpy())
            test_targets.append(y_test.cpu().numpy())

    test_predictions = np.concatenate(test_predictions, axis=0)
    test_targets = np.concatenate(test_targets, axis=0)

    test_predictions_original = scaler.inverse_transform(
        test_predictions.reshape(-1, 1)
    ).reshape(test_predictions.shape)

    test_targets_original = scaler.inverse_transform(
        test_targets.reshape(-1, 1)
    ).reshape(test_targets.shape)

    # Evaluate the last forecast step for each model
    day_pred_last = test_predictions_original[:, -1]
    day_true_last = test_targets_original[:, -1]

    rmse = mean_squared_error(day_true_last, day_pred_last, squared=False)
    mae = mean_absolute_error(day_true_last, day_pred_last)
    mape = mean_absolute_percentage_error(day_true_last, day_pred_last) * 100

    day_results[day]["rmse"].append(rmse)
    day_results[day]["mae"].append(mae)
    day_results[day]["mape"].append(mape)

    print(f"Day {day} result:")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE : {mae:.4f}")
    print(f"MAPE: {mape:.2f}%")


print("\n" + "=" * 80)
print("Final Results")
print("=" * 80)

for day in EVAL_DAYS:
    print(
        f"Day {day} - "
        f"RMSE: {np.mean(day_results[day]['rmse']):.4f}, "
        f"MAE: {np.mean(day_results[day]['mae']):.4f}, "
        f"MAPE: {np.mean(day_results[day]['mape']):.2f}%"
    )


Using device: cuda
Data: F10.7, Sequence Length: 30, Predict Length: 1
Loading model: .\SINet_1_interval_F10.pth
Loading preprocessing: ./data\F10.7_preprocess.pkl
Loading raw test csv : ./data\F10.7_test.csv
TestX shape: (4747, 30, 1)
Testy shape: (4747, 60, 1)
Raw test date range: 2008-10-04 00:00:00 to 2021-12-31 00:00:00
Day 1 result:
RMSE: 18.1898
MAE : 12.0791
MAPE: 11.28%
Data: F10.7, Sequence Length: 30, Predict Length: 27
Loading model: .\SINet_27_interval_F10.pth
Loading preprocessing: ./data\F10.7_preprocess.pkl
Loading raw test csv : ./data\F10.7_test.csv
TestX shape: (4747, 30, 1)
Testy shape: (4747, 60, 1)
Raw test date range: 2008-10-04 00:00:00 to 2021-12-31 00:00:00
Day 27 result:
RMSE: 16.7612
MAE : 11.0626
MAPE: 10.25%
Data: F10.7, Sequence Length: 30, Predict Length: 45
Loading model: .\SINet_45_interval_F10.pth
Loading preprocessing: ./data\F10.7_preprocess.pkl
Loading raw test csv : ./data\F10.7_test.csv
TestX shape: (4747, 30, 1)
Testy shape: (4747, 60, 1)
Raw te